In [1]:
import pandas as pd
import numpy as np
import time
import pickle
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import (
    accuracy_score, 
    classification_report, 
    confusion_matrix, 
    ConfusionMatrixDisplay,
    f1_score,
    make_scorer,
    log_loss
)
from sklearn.preprocessing import LabelEncoder, StandardScaler
import xgboost as xgb

def train_and_optimize():
    print("Carregando dataset...")
    try:
        df = pd.read_csv("asl_landmarks_dataset.csv")
    except FileNotFoundError:
        df = pd.read_csv("landmarks_dataset.csv")
    
    # --- OTIMIZAÇÃO 1: Remover letras que necessitam de movimento (J e Z) ---
    print("Removendo letras com movimento (J e Z)...")
    df = df[~df['label'].isin(['J', 'Z'])]
    
    # --- OTIMIZAÇÃO 2: Encoding simples na coluna 'hand' ---
    print("Realizando encoding simples na coluna 'hand'...")
    if 'hand' in df.columns:
        df['hand'] = df['hand'].replace({'Left': 0, 'Right': 1})
    elif 'hand_type' in df.columns:
        df['hand_type'] = df['hand_type'].replace({'Left': 0, 'Right': 1})
        df = df.rename(columns={'hand_type': 'hand'})

    # 1. Preparar dados
    X = df.drop(['label'], axis=1)
    y = df['label']
    
    # Label Encoder para as labels (necessário para métricas e XGBoost)
    le_label = LabelEncoder()
    y_encoded = le_label.fit_transform(y)
    
    # 2. Dividir em treino (70%), validação (15%) e teste (15%)
    X_train_val, X_test, y_train_val, y_test = train_test_split(
        X, y_encoded, test_size=0.15, random_state=42, stratify=y_encoded
    )
    X_train, X_val, y_train, y_val = train_test_split(
        X_train_val, y_train_val, test_size=0.176, random_state=42, stratify=y_train_val
    )

    # 3. Escalonamento (StandardScaler)
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_val_scaled = scaler.transform(X_val)
    X_test_scaled = scaler.transform(X_test)
    
    # 4. Configuração dos Modelos com F1-Score como métrica de otimização
    # Usamos 'f1_weighted' para lidar com possíveis desequilíbrios entre as classes
    scoring_metric = 'f1_weighted'
    
    models_config = {
        'SVM': {
            'model': SVC(probability=True, random_state=42),
            'params': {
                'C': [1, 10],
                'kernel': ['rbf', 'linear']
            }
        },
        'XGBoost': {
            'model': xgb.XGBClassifier(random_state=42, eval_metric='mlogloss'),
            'params': {
                'n_estimators': [100, 200],
                'max_depth': [3, 6],
                'learning_rate': [0.1, 0.2]
            }
        },
        'NeuralNetwork': {
            'model': MLPClassifier(random_state=42, max_iter=500),
            'params': {
                'hidden_layer_sizes': [(128, 64), (64, 64)],
                'alpha': [0.0001, 0.001]
            }
        },
        'RandomForest': {
            'model': RandomForestClassifier(random_state=42),
            'params': {
                'n_estimators': [100, 200],
                'max_depth': [None, 20]
            }
        }
    }
    
    best_overall_model = None
    best_overall_f1 = 0
    best_inf_time = float('inf')
    best_model_name = ""
    results = []

    # 5. Ciclo de Otimização
    for name, config in models_config.items():
        print(f"\nOtimizando {name} usando {scoring_metric}...")
        grid = GridSearchCV(
            config['model'], 
            config['params'], 
            cv=3, 
            n_jobs=-1, 
            verbose=1, 
            scoring=scoring_metric
        )
        
        grid.fit(X_train_scaled, y_train)
        
        # Previsões e Probabilidades para métricas avançadas
        val_preds = grid.predict(X_val_scaled)
        val_probs = grid.predict_proba(X_val_scaled)
        
        # Cálculo de métricas
        val_acc = accuracy_score(y_val, val_preds)
        val_f1 = f1_score(y_val, val_preds, average='weighted')
        val_loss = log_loss(y_val, val_probs)
        
        # Medir tempo de inferência
        start_inf = time.time()
        for _ in range(100):
            grid.predict(X_val_scaled[:1])
        inf_time = (time.time() - start_inf) / 100
        
        print(f"{name} - Val F1-Score: {val_f1:.4f}, Accuracy: {val_acc:.4f}, Log Loss: {val_loss:.4f}")
        
        results.append({
            'model_name': name,
            'val_f1': val_f1,
            'val_accuracy': val_acc,
            'inf_time': inf_time
        })
        
        # Seleção baseada no F1-Score
        if val_f1 > best_overall_f1:
            best_overall_f1 = val_f1
            best_inf_time = inf_time
            best_overall_model = grid.best_estimator_
            best_model_name = name
        elif val_f1 == best_overall_f1:
            if inf_time < best_inf_time:
                best_inf_time = inf_time
                best_overall_model = grid.best_estimator_
                best_model_name = name

    # 6. Avaliação Final no Teste
    print(f"\n--- AVALIAÇÃO FINAL (Melhor Modelo: {best_model_name}) ---")
    test_preds = best_overall_model.predict(X_test_scaled)
    print(classification_report(y_test, test_preds, target_names=le_label.classes_))
    
    # 7. Guardar ficheiros
    with open('melhor_modelo.pkl', 'wb') as f:
        pickle.dump(best_overall_model, f)
    with open('scaler.pkl', 'wb') as f:
        pickle.dump(scaler, f)
    with open('label_encoder.pkl', 'wb') as f:
        pickle.dump(le_label, f)
    
    print(f"Treino concluído. Ficheiros guardados com sucesso.")

if __name__ == "__main__":
    df = pd.read_csv("asl_landmarks_dataset.csv")

    #train_and_optimize()
    #pass

Carregando dataset...
Removendo letras com movimento (J e Z)...
Realizando encoding simples na coluna 'hand'...

Otimizando XGBoost usando f1_weighted...
Fitting 3 folds for each of 8 candidates, totalling 24 fits


C:\Users\pedrodgoncalves\AppData\Local\Temp\ipykernel_25316\436438267.py:39: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df['hand'] = df['hand'].replace({'Left': 0, 'Right': 1})


XGBoost - Val F1-Score: 0.9983, Accuracy: 0.9983, Log Loss: 0.0099

--- AVALIAÇÃO FINAL (Melhor Modelo: XGBoost) ---
              precision    recall  f1-score   support

           A       1.00      1.00      1.00       150
           B       1.00      1.00      1.00       150
           C       1.00      1.00      1.00       142
           D       1.00      1.00      1.00       150
           E       1.00      1.00      1.00       150
           F       1.00      0.99      1.00       150
           G       0.99      1.00      1.00       150
           H       1.00      0.99      1.00       150
           I       1.00      0.99      1.00       150
           K       1.00      1.00      1.00       150
           L       1.00      1.00      1.00       150
           M       0.99      1.00      1.00       141
           N       1.00      1.00      1.00       150
           O       1.00      1.00      1.00       103
           P       1.00      1.00      1.00       150
           Q      

In [2]:
df = pd.read_csv("asl_landmarks_dataset.csv")

In [6]:

import pandas as pd
import numpy as np
import time
import pickle
import argparse
import warnings

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    f1_score,
    log_loss
)

from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier

import xgboost as xgb

warnings.filterwarnings("ignore")


# -----------------------------
#  PREPROCESSAMENTO
# -----------------------------

def preprocess_landmarks(
    df: pd.DataFrame,
    drop_motion_letters=True,
    remove_out_of_range=True,
    mirror_left_to_right=True,
    normalize_center_scale=True,
    keep_hand_feature=True
) -> pd.DataFrame:
    """
    Preprocessamento específico para landmarks MediaPipe Hands:
    - remove J e Z (opcional)
    - remove amostras com XY fora [0,1] (opcional)
    - espelha mão esquerda (x = 1-x) para unificar referencial (opcional)
    - centra no WRIST e escala pelo tamanho (WRIST -> MIDDLE_FINGER_MCP) (opcional)
    - converte 'hand' em feature binária "was_left" (opcional)
    """

    df = df.copy()

    # (0) Remover letras com movimento (J e Z)
    if drop_motion_letters and "label" in df.columns:
        df = df[~df["label"].isin(["J", "Z"])].reset_index(drop=True)

    # Identificar colunas de landmarks
    lm_cols = [c for c in df.columns if c.endswith(("_x", "_y", "_z"))]
    x_cols = [c for c in lm_cols if c.endswith("_x")]
    y_cols = [c for c in lm_cols if c.endswith("_y")]
    z_cols = [c for c in lm_cols if c.endswith("_z")]

    # (1) Remover amostras com XY fora de [0,1] (ruído do detector)
    if remove_out_of_range and len(x_cols) > 0 and len(y_cols) > 0:
        xy_cols = x_cols + y_cols
        mask_ok = ~((df[xy_cols] < 0) | (df[xy_cols] > 1)).any(axis=1)
        removed = int((~mask_ok).sum())
        if removed > 0:
            print(f"[Preprocess] Removidas {removed} amostras com XY fora de [0,1].")
        df = df.loc[mask_ok].reset_index(drop=True)

    # (2) Espelhar mão esquerda para o referencial da direita
    was_left = None
    if "hand" in df.columns:
        was_left = df["hand"].astype(str).str.lower().eq("left").values

        if mirror_left_to_right and len(x_cols) > 0:
            # Mirror no espaço normalizado [0,1]: x := 1 - x
            df.loc[was_left, x_cols] = 1.0 - df.loc[was_left, x_cols]

        # Converter hand em feature binária (opcional)
        if keep_hand_feature:
            df["was_left"] = was_left.astype(int)

        # remover coluna original de texto para evitar problemas no scaler/modelos
        df = df.drop(columns=["hand"])

    # (3) Normalização: centrar no WRIST e escalar por WRIST->MIDDLE_FINGER_MCP
    if normalize_center_scale:
        required = ["WRIST_x", "WRIST_y", "WRIST_z",
                    "MIDDLE_FINGER_MCP_x", "MIDDLE_FINGER_MCP_y", "MIDDLE_FINGER_MCP_z"]
        missing_req = [c for c in required if c not in df.columns]
        if missing_req:
            raise ValueError(f"Colunas necessárias para normalização não encontradas: {missing_req}")

        # Centrar: subtrair WRIST
        df[x_cols] = df[x_cols].sub(df["WRIST_x"], axis=0)
        df[y_cols] = df[y_cols].sub(df["WRIST_y"], axis=0)
        df[z_cols] = df[z_cols].sub(df["WRIST_z"], axis=0)

        # Escalar: distância WRIST -> MIDDLE_FINGER_MCP (após centrar, WRIST=0)
        scale = np.sqrt(
            df["MIDDLE_FINGER_MCP_x"] ** 2 +
            df["MIDDLE_FINGER_MCP_y"] ** 2 +
            df["MIDDLE_FINGER_MCP_z"] ** 2
        )

        # Evitar divisões por zero (casos raros)
        med = float(np.nanmedian(scale.values))
        scale = scale.replace(0, np.nan).fillna(med if med > 0 else 1.0)

        df[lm_cols] = df[lm_cols].div(scale, axis=0)

    return df


# -----------------------------
#  TREINO + OTIMIZAÇÃO
# -----------------------------

def train_and_optimize(
    csv_path="asl_landmarks_dataset.csv",
    output_prefix="melhor_modelo",
    drop_motion_letters=True,
    calibrate=False
):
    print("Carregando dataset...")
    df = pd.read_csv(csv_path)
    print(f"Dataset carregado: {df.shape}")

    # Preprocessamento robusto
    print("\n[1/6] Preprocessamento (outliers XY, mirror Left->Right, normalização)...")
    df = preprocess_landmarks(
        df,
        drop_motion_letters=drop_motion_letters,
        remove_out_of_range=True,
        mirror_left_to_right=True,
        normalize_center_scale=True,
        keep_hand_feature=True  # cria coluna was_left
    )
    print(f"Após preprocess: {df.shape}")

    # Preparar dados
    if "label" not in df.columns:
        raise ValueError("Coluna 'label' não encontrada no dataset.")

    X = df.drop(columns=["label"])
    y = df["label"]

    # LabelEncoder (para XGBoost e métricas)
    le_label = LabelEncoder()
    y_encoded = le_label.fit_transform(y)

    # Split: treino (70%), validação (15%), teste (15%)
    print("\n[2/6] Split treino/val/teste...")
    X_train_val, X_test, y_train_val, y_test = train_test_split(
        X, y_encoded, test_size=0.15, random_state=42, stratify=y_encoded
    )
    X_train, X_val, y_train, y_val = train_test_split(
        X_train_val, y_train_val, test_size=0.176, random_state=42, stratify=y_train_val
    )

    # Configuração CV
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

    scoring_metric = "f1_weighted"

    # Pipelines (scaler dentro do pipeline => evita leakage)
    # Obs: árvores não precisam de scaler, mas manter no pipeline simplifica e não atrapalha muito.
    models_config = {
        "SVM": {
            "pipeline": Pipeline([
                ("scaler", StandardScaler()),
                ("clf", SVC(probability=True, class_weight="balanced", random_state=42))
            ]),
            "params": {
                "clf__C": [1, 10, 30],
                "clf__kernel": ["rbf", "linear"],
                "clf__gamma": ["scale", "auto"]
            }
        },
        "XGBoost": {
            "pipeline": Pipeline([
                ("scaler", StandardScaler()),
                ("clf", xgb.XGBClassifier(
                    random_state=42,
                    eval_metric="mlogloss",
                    tree_method="hist",
                    n_jobs=-1
                ))
            ]),
            "params": {
                "clf__n_estimators": [300, 600],
                "clf__max_depth": [3, 6],
                "clf__learning_rate": [0.05, 0.1],
                "clf__subsample": [0.8, 1.0],
                "clf__colsample_bytree": [0.8, 1.0]
            }
        },
        "NeuralNetwork": {
            "pipeline": Pipeline([
                ("scaler", StandardScaler()),
                ("clf", MLPClassifier(
                    random_state=42,
                    max_iter=800,
                    early_stopping=True,
                    n_iter_no_change=20
                ))
            ]),
            "params": {
                "clf__hidden_layer_sizes": [(256, 128), (128, 64), (128, 128)],
                "clf__alpha": [0.0005, 0.001, 0.005],
                "clf__learning_rate_init": [0.001, 0.0005]
            }
        },
        "RandomForest": {
            "pipeline": Pipeline([
                # scaler não é necessário, mas fica para manter consistência do pipeline
                ("scaler", StandardScaler()),
                ("clf", RandomForestClassifier(
                    random_state=42,
                    class_weight="balanced_subsample",
                    n_jobs=-1
                ))
            ]),
            "params": {
                "clf__n_estimators": [300, 600],
                "clf__max_depth": [None, 25],
                "clf__min_samples_split": [2, 5],
                "clf__min_samples_leaf": [1, 2]
            }
        }
    }

    best_overall_model = None
    best_overall_f1 = -1
    best_inf_time = float("inf")
    best_model_name = ""
    results = []

    # Treino/otimização
    print("\n[3/6] GridSearchCV (isto pode demorar)...")
    for name, cfg in models_config.items():
        print(f"\n--- Otimizando {name} usando {scoring_metric} ---")
        grid = GridSearchCV(
            estimator=cfg["pipeline"],
            param_grid=cfg["params"],
            cv=cv,
            n_jobs=-1,
            verbose=1,
            scoring=scoring_metric
        )

        start_train = time.time()
        grid.fit(X_train, y_train)
        train_time = time.time() - start_train

        # Métricas na validação
        val_preds = grid.predict(X_val)
        if hasattr(grid.best_estimator_, "predict_proba"):
            val_probs = grid.predict_proba(X_val)
            val_loss = log_loss(y_val, val_probs)
        else:
            val_probs = None
            val_loss = np.nan

        val_acc = accuracy_score(y_val, val_preds)
        val_f1 = f1_score(y_val, val_preds, average="weighted")

        # Tempo inferência (média)
        start_inf = time.time()
        for _ in range(200):
            _ = grid.predict(X_val.iloc[:1])
        inf_time = (time.time() - start_inf) / 200.0

        print(f"{name} | Val F1: {val_f1:.4f} | Acc: {val_acc:.4f} | LogLoss: {val_loss:.4f} "
              f"| TrainTime: {train_time:.1f}s | Inf: {inf_time*1000:.3f} ms")

        results.append({
            "model_name": name,
            "val_f1": val_f1,
            "val_accuracy": val_acc,
            "val_logloss": val_loss,
            "inf_time_ms": inf_time * 1000.0,
            "best_params": grid.best_params_
        })

Shape: (25566, 65)

Colunas: ['hand', 'WRIST_x', 'WRIST_y', 'WRIST_z', 'THUMB_CMC_x', 'THUMB_CMC_y', 'THUMB_CMC_z', 'THUMB_MCP_x', 'THUMB_MCP_y', 'THUMB_MCP_z', 'THUMB_IP_x', 'THUMB_IP_y', 'THUMB_IP_z', 'THUMB_TIP_x', 'THUMB_TIP_y'] ...

Missing total: 0
Top missing cols:
 hand           0
WRIST_x        0
WRIST_y        0
WRIST_z        0
THUMB_CMC_x    0
THUMB_CMC_y    0
THUMB_CMC_z    0
THUMB_MCP_x    0
THUMB_MCP_y    0
THUMB_MCP_z    0
THUMB_IP_x     0
THUMB_IP_y     0
THUMB_IP_z     0
THUMB_TIP_x    0
THUMB_TIP_y    0
dtype: int64
Duplicados: 0

Labels:
 label
A    1000
D    1000
I    1000
E    1000
F    1000
H    1000
P    1000
Q    1000
L    1000
K    1000
W    1000
X    1000
Y    1000
V    1000
R    1000
U    1000
T    1000
S    1000
Z    1000
N     999
J     999
B     998
G     997
C     947
M     939
O     687
Name: count, dtype: int64

Hand:
 hand
Left     21377
Right     4189
Name: count, dtype: int64

N landmark cols: 63

Range global landmarks:
min: -0.6197940111160278 ma

## Accuracy (Acurácia)

<center>
<span style="font-size: 24px;">
$\text{Accuracy} = \frac{\text{nº de previsões corretas}}{\text{nº total de previsões}}$
</span>
</center>


A métrica principal utilizada foi a acurácia, uma vez que o problema consiste numa classificação multiclasse com classes equilibradas, onde todas as letras têm igual importância.

    models_config = {
        'SVM': {
            'model': SVC(probability=True, random_state=42),
            'params': {
                'C': [1, 10],
                'kernel': ['rbf', 'linear']
            }
        },
        'XGBoost': {
            'model': xgb.XGBClassifier(random_state=42, eval_metric='mlogloss'),
            'params': {
                'n_estimators': [100, 200],
                'max_depth': [3, 6],
                'learning_rate': [0.1, 0.2]
            }
        },
        'NeuralNetwork': {
            'model': MLPClassifier(random_state=42, max_iter=500),
            'params': {
                'hidden_layer_sizes': [(128, 64), (64, 64)],
                'alpha': [0.0001, 0.001]
            }
        },
        'RandomForest': {
            'model': RandomForestClassifier(random_state=42),
            'params': {
                'n_estimators': [100, 200],
                'max_depth': [None, 20]
            }
        }
    }